# Sentinel — FHIR Observation Data Exploration

This notebook explores the historical Synthea FHIR `Observation` data used by Sentinel. The goal is to understand the raw FHIR value structures, verify the normalization logic used to build the Silver layer, and explore heart-rate windows for baseline profiling.

> **Scope:** exploratory analysis only. Production transformation and streaming logic live in the `sentinel` package.


In [1]:
from collections import Counter

from sentinel.ingestion.gcs import read_ndjson


# FHIR Observation fields that may contain the actual observed value.
VALUE_FIELDS = [
    "valueQuantity",
    "valueCodeableConcept",
    "valueString",
    "valueBoolean",
    "valueInteger",
    "valueRange",
    "valueRatio",
    "valueDateTime",
    "valuePeriod",
]


counts = Counter()

for obs in read_ndjson("Observation"):

    found = False

    for field in VALUE_FIELDS:
        if field in obs:
            counts[field] += 1
            found = True

    if "component" in obs:
        counts["component"] += 1
        found = True

    if not found:
        counts["no_value_found"] += 1


for field, count in counts.most_common():
    print(f"{field:<25} {count:>8}")


valueQuantity                64672
valueCodeableConcept         15652
component                     3310
valueString                     49


## 1. Explore raw FHIR Observation value structures

FHIR Observations can store values in several different fields. Start by counting which value representations occur in the historical dataset.


In [2]:
import json

examples = {}

for obs in read_ndjson("Observation"):

    fields = VALUE_FIELDS + ["component"]

    for field in fields:
        if field in obs and field not in examples:
            examples[field] = obs

    if len(examples) == len(fields):
        break


for field, obs in examples.items():
    print("\n" + "=" * 80)
    print(field)
    print("=" * 80)

    print(
        json.dumps(
            obs,
            indent=2,
            ensure_ascii=False
        )
    )



valueQuantity
{
  "resourceType": "Observation",
  "id": "60892e56-6ba4-b166-bc8a-6b3a3ab67b96",
  "meta": {
    "profile": [
      "http://hl7.org/fhir/us/core/StructureDefinition/us-core-body-height"
    ]
  },
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "vital-signs",
          "display": "Vital signs"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "8302-2",
        "display": "Body Height"
      }
    ],
    "text": "Body Height"
  },
  "subject": {
    "reference": "Patient/60892e56-6ba4-b166-6b50-36314eb175dc"
  },
  "encounter": {
    "reference": "Encounter/60892e56-6ba4-b166-ec60-913e925fea8c"
  },
  "effectiveDateTime": "2017-11-29T11:43:42+03:00",
  "issued": "2017-11-29T11:43:42.928+03:00",
  "valueQuantity": {
    "value": 52.4,
    "unit": "cm",
    "system": "http://unitsofm

### Inspect one example of each value structure

Looking at representative raw resources helps confirm what the normalizer needs to support.


In [3]:
# Components can have their own value[x] fields, so profile them separately.
component_types = Counter()

for obs in read_ndjson("Observation"):

    for component in obs.get("component", []):

        for field in VALUE_FIELDS:
            if field in component:
                component_types[field] += 1


for field, count in component_types.most_common():
    print(f"{field:<25} {count:>8}")


valueCodeableConcept         25866
valueQuantity                 6617
valueString                   1447


### Inspect values nested inside `component`

Some FHIR Observations are panels. Their measurements are stored as components rather than directly on the parent Observation.


In [4]:
from collections import defaultdict

codes_by_type = defaultdict(Counter)

for obs in read_ndjson("Observation"):

    coding = obs.get("code", {}).get("coding", [])

    code = coding[0].get("code") if coding else None
    name = obs.get("code", {}).get("text")

    for field in VALUE_FIELDS:
        if field in obs:
            codes_by_type[field][(code, name)] += 1

    if "component" in obs:
        codes_by_type["component"][(code, name)] += 1


for value_type, codes in codes_by_type.items():

    print(f"\n--- {value_type} ---")

    for (code, name), count in codes.most_common(10):
        print(f"{str(code):<12} {count:>6}  {name}")



--- valueQuantity ---
72514-3        3449  Pain severity - 0-10 verbal numeric rating [Score] - Reported
33914-3        2048  Glomerular filtration rate [Volume Rate/Area] in Serum or Plasma by Creatinine-based formula (MDRD)/1.73 sq M
29463-7        1861  Body Weight
8867-4         1792  Heart rate
9279-1         1792  Respiratory rate
8302-2         1761  Body Height
39156-5        1668  Body mass index (BMI) [Ratio]
2339-0         1474  Glucose [Mass/volume] in Blood
38483-4        1474  Creatinine [Mass/volume] in Blood
49765-1        1474  Calcium [Mass/volume] in Blood

--- component ---
85354-9        1847  Blood pressure panel with all children optional
93025-5        1437  Protocol for Responding to and Assessing Patients' Assets, Risks, and Experiences [PRAPARE]
X9999-0          10  Operative Status
91148-7           8  Pain intensity, Enjoyment of life, General activity scale [PEG]
85354-9           7  Blood Pressure panel with all children optional
86923-0           1  Kan

### Identify common observation codes by value type

This shows which clinical signals dominate each FHIR representation and helps prioritize normalization coverage.


In [5]:

for obs in read_ndjson("Observation"):
    coding = obs.get("code", {}).get("coding", [])

    if any(c.get("code") == "72166-2" for c in coding):
        print(json.dumps(obs, indent=2))
        break


{
  "resourceType": "Observation",
  "id": "60892e56-6ba4-b166-2a7f-e629d5e3add7",
  "meta": {
    "profile": [
      "http://hl7.org/fhir/us/core/StructureDefinition/us-core-smokingstatus",
      "http://hl7.org/fhir/us/core/StructureDefinition/us-core-simple-observation"
    ]
  },
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "social-history",
          "display": "Social history"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "72166-2",
        "display": "Tobacco smoking status"
      }
    ],
    "text": "Tobacco smoking status"
  },
  "subject": {
    "reference": "Patient/60892e56-6ba4-b166-6b50-36314eb175dc"
  },
  "encounter": {
    "reference": "Encounter/60892e56-6ba4-b166-ec60-913e925fea8c"
  },
  "effectiveDateTime": "2017-11-29T11:43:42+03:00",
  "issued": "2017-11-29T11:43:42.

## 2. Inspect representative FHIR cases

### Categorical observation: smoking status

Smoking status is useful for verifying `valueCodeableConcept` handling.


In [6]:
for obs in read_ndjson("Observation"):

    for component in obs.get("component", []):

        if "valueCodeableConcept" in component:
            print("PARENT OBSERVATION:")
            print(
                obs.get("code", {}).get("text")
            )

            print("\nCOMPONENT:")
            print(
                json.dumps(
                    component,
                    indent=2
                )
            )

            break
    else:
        continue

    break


PARENT OBSERVATION:
Protocol for Responding to and Assessing Patients' Assets, Risks, and Experiences [PRAPARE]

COMPONENT:
{
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "76501-6",
        "display": "Within the last year, have you been afraid of your partner or ex-partner"
      }
    ],
    "text": "Within the last year, have you been afraid of your partner or ex-partner"
  },
  "valueCodeableConcept": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "LA32-8",
        "display": "No"
      }
    ],
    "text": "No"
  }
}


### Categorical component inside a panel

This example checks that categorical values can also appear inside `component`.


In [7]:
from sentinel.ingestion.gcs import read_ndjson
from sentinel.transformation.observations import normalize_observation


for obs in read_ndjson("Observation"):
    rows = normalize_observation(obs)

    for row in rows:
        print(row)

    break


{'observation_id': '60892e56-6ba4-b166-bc8a-6b3a3ab67b96', 'patient_id': '60892e56-6ba4-b166-6b50-36314eb175dc', 'encounter_id': '60892e56-6ba4-b166-ec60-913e925fea8c', 'parent_code': '8302-2', 'parent_display': 'Body Height', 'event_time': '2017-11-29T11:43:42+03:00', 'issued_time': '2017-11-29T11:43:42.928+03:00', 'status': 'final', 'code': '8302-2', 'display': 'Body Height', 'value_type': 'numeric', 'numeric_value': 52.4, 'categorical_code': None, 'categorical_display': None, 'string_value': None, 'unit': 'cm'}


## 3. Validate the Observation normalizer

First, normalize a single Observation to verify the general output shape.


In [9]:
from sentinel.transformation.observations import get_code_info, normalize_observation

for obs in read_ndjson("Observation"):

    code, _ = get_code_info(obs.get("code", {}))

    if code == "85354-9":  # LOINC blood-pressure panel

        rows = normalize_observation(obs)

        for row in rows:
            print(row)

        break


{'observation_id': '60892e56-6ba4-b166-f4be-24f1f9d51441', 'patient_id': '60892e56-6ba4-b166-6b50-36314eb175dc', 'encounter_id': '60892e56-6ba4-b166-ec60-913e925fea8c', 'parent_code': '85354-9', 'parent_display': 'Blood pressure panel with all children optional', 'event_time': '2017-11-29T11:43:42+03:00', 'issued_time': '2017-11-29T11:43:42.928+03:00', 'status': 'final', 'code': '8462-4', 'display': 'Diastolic Blood Pressure', 'value_type': 'numeric', 'numeric_value': 78, 'categorical_code': None, 'categorical_display': None, 'string_value': None, 'unit': 'mm[Hg]'}
{'observation_id': '60892e56-6ba4-b166-f4be-24f1f9d51441', 'patient_id': '60892e56-6ba4-b166-6b50-36314eb175dc', 'encounter_id': '60892e56-6ba4-b166-ec60-913e925fea8c', 'parent_code': '85354-9', 'parent_display': 'Blood pressure panel with all children optional', 'event_time': '2017-11-29T11:43:42+03:00', 'issued_time': '2017-11-29T11:43:42.928+03:00', 'status': 'final', 'code': '8480-6', 'display': 'Systolic Blood Pressure'

### Multi-component example: blood pressure

A blood-pressure panel should expand from one parent FHIR Observation into separate normalized measurements.


In [10]:
from sentinel.ingestion.gcs import read_ndjson
from sentinel.transformation.observations import (
    get_code_info,
    normalize_observation,
)

for obs in read_ndjson("Observation"):
    code, _ = get_code_info(obs.get("code", {}))

    if code == "85354-9":
        rows = normalize_observation(obs)

        print(f"Number of Silver rows: {len(rows)}")

        for row in rows:
            print(
                row["parent_code"],
                "→",
                row["code"],
                row["display"],
                "=",
                row["numeric_value"],
                row["unit"],
            )

        break


Number of Silver rows: 2
85354-9 → 8462-4 Diastolic Blood Pressure = 78 mm[Hg]
85354-9 → 8480-6 Systolic Blood Pressure = 128 mm[Hg]


In [11]:
from sentinel.ingestion.gcs import read_ndjson
from sentinel.transformation.observations import (
    get_code_info,
    normalize_observation,
)

for obs in read_ndjson("Observation"):
    code, _ = get_code_info(obs.get("code", {}))

    if code == "72166-2":  # Tobacco smoking status
        rows = normalize_observation(obs)

        for row in rows:
            print(row)

        break


{'observation_id': '60892e56-6ba4-b166-2a7f-e629d5e3add7', 'patient_id': '60892e56-6ba4-b166-6b50-36314eb175dc', 'encounter_id': '60892e56-6ba4-b166-ec60-913e925fea8c', 'parent_code': '72166-2', 'parent_display': 'Tobacco smoking status', 'event_time': '2017-11-29T11:43:42+03:00', 'issued_time': '2017-11-29T11:43:42.928+03:00', 'status': 'final', 'code': '72166-2', 'display': 'Tobacco smoking status', 'value_type': 'categorical', 'numeric_value': None, 'categorical_code': '266919005', 'categorical_display': 'Never smoked tobacco (finding)', 'string_value': None, 'unit': None}


### Categorical normalization

Verify that a categorical Observation is preserved correctly in the normalized representation.


In [12]:
import polars as pl

df = pl.read_parquet(
    "gs://sentinel-fhir-data/silver/observations/run_id=001/observations.parquet"
)

print(
    df.select(
        pl.col("event_time").min().alias("earliest"),
        pl.col("event_time").max().alias("latest"),
        pl.len().alias("rows"),
        pl.col("patient_id").n_unique().alias("patients"),
    )
)


shape: (1, 4)
┌───────────────────┬───────────────────┬────────┬──────────┐
│ earliest          ┆ latest            ┆ rows   ┆ patients │
│ ---               ┆ ---               ┆ ---    ┆ ---      │
│ datetime[μs, UTC] ┆ datetime[μs, UTC] ┆ u32    ┆ u32      │
╞═══════════════════╪═══════════════════╪════════╪══════════╡
│ null              ┆ null              ┆ 114303 ┆ 119      │
└───────────────────┴───────────────────┴────────┴──────────┘


### Verify raw FHIR timestamps

This quick check was used to confirm the source timestamp format during development.


In [16]:
observation = next(read_ndjson("Observation"))

print(observation["effectiveDateTime"])
print(observation["issued"])


2017-11-29T11:43:42+03:00
2017-11-29T11:43:42.928+03:00


## 5. Explore candidate baseline window sizes

Before choosing a baseline strategy, estimate how many fixed-record windows the historical dataset can support.


In [18]:
import polars as pl

df = pl.read_parquet(
    "gs://sentinel-fhir-data/"
    "silver/observations/run_id=001/observations.parquet"
)

print(f"Total Silver rows: {df.height:,}")

# Compare several fixed-record window sizes before choosing one.
for records_per_window in [50, 100, 250, 500, 1000]:
    windows = df.height // records_per_window

    print(
        f"{records_per_window:>4} records/window "
        f"→ {windows:,} possible windows"
    )


Total Silver rows: 114,303
  50 records/window → 2,286 possible windows
 100 records/window → 1,143 possible windows
 250 records/window → 457 possible windows
 500 records/window → 228 possible windows
1000 records/window → 114 possible windows


### Focus on heart rate

Heart rate (`LOINC 8867-4`) is used as an initial numeric signal for baseline experiments.


In [19]:
heart_rate = df.filter(
    pl.col("code") == "8867-4"
)

print(f"\nHeart-rate records: {heart_rate.height:,}")

for records_per_window in [10, 20, 50, 100]:
    windows = heart_rate.height // records_per_window

    print(
        f"{records_per_window:>3} HR records/window "
        f"→ {windows:,} possible windows"
    )



Heart-rate records: 1,792
 10 HR records/window → 179 possible windows
 20 HR records/window → 89 possible windows
 50 HR records/window → 35 possible windows
100 HR records/window → 17 possible windows


## 6. Profile heart-rate windows

Start with a 20-record heart-rate window and calculate its basic quality and distribution metrics.


In [20]:
import polars as pl

from sentinel.baseline.profiler import profile_numeric_window


df = pl.read_parquet(
    "gs://sentinel-fhir-data/"
    "silver/observations/run_id=001/observations.parquet"
)

heart_rate = (
    df
    .filter(pl.col("code") == "8867-4")
    .sort("event_time")
)

first_window = heart_rate.head(20)

profile = profile_numeric_window(first_window)

profile


{'record_count': 20,
 'null_count': 0,
 'null_rate': 0.0,
 'mean': 82.75,
 'std': 12.030115718916687,
 'median': 84.0,
 'p25': 73.0,
 'p75': 91.0}

### Create all 20-record heart-rate windows

The historical windows below are for baseline exploration. They are separate from the live processing-time windows used by the streaming pipeline.


In [21]:
from sentinel.baseline.windows import create_signal_windows

# Historical fixed-record windows used only for baseline exploration.
windows = create_signal_windows(
    df,
    code="8867-4",
    window_size=20,
)

print(f"Number of windows: {len(windows)}")
print(f"Rows in first window: {windows[0].height}")
print(f"Rows in last window: {windows[-1].height}")


Number of windows: 89
Rows in first window: 20
Rows in last window: 20


### Profile every historical window

Calculate the same metrics for each window so we can study normal variation across the historical data.


In [22]:
from sentinel.baseline.profiler import profile_numeric_window

profiles = []

for window_id, window in enumerate(windows):
    profile = profile_numeric_window(window)

    profile["window_id"] = window_id

    profiles.append(profile)

profiles_df = pl.DataFrame(profiles)

print(profiles_df)


shape: (89, 9)
┌──────────────┬────────────┬───────────┬──────────┬───┬────────┬──────┬──────┬───────────┐
│ record_count ┆ null_count ┆ null_rate ┆ mean     ┆ … ┆ median ┆ p25  ┆ p75  ┆ window_id │
│ ---          ┆ ---        ┆ ---       ┆ ---      ┆   ┆ ---    ┆ ---  ┆ ---  ┆ ---       │
│ i64          ┆ i64        ┆ f64       ┆ f64      ┆   ┆ f64    ┆ f64  ┆ f64  ┆ i64       │
╞══════════════╪════════════╪═══════════╪══════════╪═══╪════════╪══════╪══════╪═══════════╡
│ 20           ┆ 0          ┆ 0.0       ┆ 82.75    ┆ … ┆ 84.0   ┆ 73.0 ┆ 91.0 ┆ 0         │
│ 20           ┆ 0          ┆ 0.0       ┆ 78.9     ┆ … ┆ 79.5   ┆ 68.0 ┆ 90.0 ┆ 1         │
│ 20           ┆ 0          ┆ 0.0       ┆ 81.55    ┆ … ┆ 80.5   ┆ 73.0 ┆ 90.0 ┆ 2         │
│ 20           ┆ 0          ┆ 0.0       ┆ 77.85    ┆ … ┆ 76.5   ┆ 66.0 ┆ 90.0 ┆ 3         │
│ 20           ┆ 0          ┆ 0.0       ┆ 83.25    ┆ … ┆ 86.5   ┆ 73.0 ┆ 95.0 ┆ 4         │
│ …            ┆ …          ┆ …         ┆ …        ┆ … ┆ …      ┆

### Summarize variation across windows

The distribution of window-level metrics gives us context for defining a healthy baseline later.


In [23]:
print(
    profiles_df.select(
        "null_rate",
        "mean",
        "std",
        "median",
        "p25",
        "p75",
    ).describe()
)


shape: (9, 7)
┌────────────┬───────────┬──────────┬───────────┬───────────┬───────────┬──────────┐
│ statistic  ┆ null_rate ┆ mean     ┆ std       ┆ median    ┆ p25       ┆ p75      │
│ ---        ┆ ---       ┆ ---      ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str        ┆ f64       ┆ f64      ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞════════════╪═══════════╪══════════╪═══════════╪═══════════╪═══════════╪══════════╡
│ count      ┆ 89.0      ┆ 89.0     ┆ 89.0      ┆ 89.0      ┆ 89.0      ┆ 89.0     │
│ null_count ┆ 0.0       ┆ 0.0      ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ mean       ┆ 0.0       ┆ 80.56501 ┆ 12.568221 ┆ 80.342697 ┆ 71.269663 ┆ 89.02036 │
│ std        ┆ 0.0       ┆ 4.245829 ┆ 4.906094  ┆ 4.41369   ┆ 3.845789  ┆ 5.116034 │
│ min        ┆ 0.0       ┆ 73.0     ┆ 9.236512  ┆ 70.0      ┆ 63.0      ┆ 75.0     │
│ 25%        ┆ 0.0       ┆ 78.25    ┆ 10.671655 ┆ 77.5      ┆ 69.0      ┆ 86.0     │
│ 50%        ┆ 0.0       ┆ 80.3     ┆ 11.380963 ┆ 8

## 7. Investigate unusual historical windows

Inspect the window with the highest mean heart rate rather than automatically treating it as bad data.


In [24]:
max_mean_row = (
    profiles_df
    .sort("mean", descending=True)
    .head(1)
)

print(max_mean_row)


shape: (1, 9)
┌──────────────┬────────────┬───────────┬──────────┬───┬────────┬──────┬────────┬───────────┐
│ record_count ┆ null_count ┆ null_rate ┆ mean     ┆ … ┆ median ┆ p25  ┆ p75    ┆ window_id │
│ ---          ┆ ---        ┆ ---       ┆ ---      ┆   ┆ ---    ┆ ---  ┆ ---    ┆ ---       │
│ i64          ┆ i64        ┆ f64       ┆ f64      ┆   ┆ f64    ┆ f64  ┆ f64    ┆ i64       │
╞══════════════╪════════════╪═══════════╪══════════╪═══╪════════╪══════╪════════╪═══════════╡
│ 20           ┆ 0          ┆ 0.0       ┆ 105.1611 ┆ … ┆ 86.5   ┆ 76.0 ┆ 115.01 ┆ 44        │
└──────────────┴────────────┴───────────┴──────────┴───┴────────┴──────┴────────┴───────────┘


In [25]:
max_window_id = max_mean_row["window_id"][0]

print(
    windows[max_window_id].select(
        "observation_id",
        "patient_id",
        "event_time",
        "numeric_value",
        "unit",
    )
)


shape: (20, 5)
┌──────────────────────────┬──────────────────────────┬─────────────────────┬───────────────┬──────┐
│ observation_id           ┆ patient_id               ┆ event_time          ┆ numeric_value ┆ unit │
│ ---                      ┆ ---                      ┆ ---                 ┆ ---           ┆ ---  │
│ str                      ┆ str                      ┆ datetime[μs, UTC]   ┆ f64           ┆ str  │
╞══════════════════════════╪══════════════════════════╪═════════════════════╪═══════════════╪══════╡
│ 50ded46d-42cc-78e7-c694- ┆ 50ded46d-42cc-78e7-7d07- ┆ 2020-02-11 12:38:44 ┆ 68.0          ┆ /min │
│ 8aa0ae…                  ┆ 1f94db…                  ┆ UTC                 ┆               ┆      │
│ 9ab0c1c4-b9a5-5e52-6a0f- ┆ 9ab0c1c4-b9a5-5e52-a44d- ┆ 2020-02-12 11:47:40 ┆ 76.0          ┆ /min │
│ 80652c…                  ┆ 2b3534…                  ┆ UTC                 ┆               ┆      │
│ c590d962-e0c3-7798-3ca2- ┆ c590d962-e0c3-7798-82e3- ┆ 2020-02-16 16:19:45 

### Inspect the window with the highest variability

A high standard deviation can reveal a heterogeneous or unusual window and is worth inspecting before baseline thresholds are chosen.


In [26]:
max_std_row = (
    profiles_df
    .sort("std", descending=True)
    .head(1)
)

print(max_std_row)

max_std_window_id = max_std_row["window_id"][0]

print(
    windows[max_std_window_id].select(
        "patient_id",
        "event_time",
        "numeric_value",
        "unit",
    )
)


shape: (1, 9)
┌──────────────┬────────────┬───────────┬──────────┬───┬────────┬──────┬────────┬───────────┐
│ record_count ┆ null_count ┆ null_rate ┆ mean     ┆ … ┆ median ┆ p25  ┆ p75    ┆ window_id │
│ ---          ┆ ---        ┆ ---       ┆ ---      ┆   ┆ ---    ┆ ---  ┆ ---    ┆ ---       │
│ i64          ┆ i64        ┆ f64       ┆ f64      ┆   ┆ f64    ┆ f64  ┆ f64    ┆ i64       │
╞══════════════╪════════════╪═══════════╪══════════╪═══╪════════╪══════╪════════╪═══════════╡
│ 20           ┆ 0          ┆ 0.0       ┆ 105.1611 ┆ … ┆ 86.5   ┆ 76.0 ┆ 115.01 ┆ 44        │
└──────────────┴────────────┴───────────┴──────────┴───┴────────┴──────┴────────┴───────────┘
shape: (20, 4)
┌─────────────────────────────────┬─────────────────────────┬───────────────┬──────┐
│ patient_id                      ┆ event_time              ┆ numeric_value ┆ unit │
│ ---                             ┆ ---                     ┆ ---           ┆ ---  │
│ str                             ┆ datetime[μs, UTC]     